**BfPortf 04 Sequential time series splitting**

This example shows a walk-forward validation of the search. The history is cut into consecutive segments: every
segment has its own train window (the portfolio is found there) and its own test window (the portfolio is checked
there). The out-of-sample returns of all segments are stitched into a single track record and compared with the
SPY benchmark.

The split itself is a small function with two parameters (`train_days` and `test_days`), the search and the reports
come from `bfportf`. Example `01` shows a single train/test split, example `03` goes through the metrics and the
strategies in detail.

The examples need the `examples` extra (`uv sync --extra examples`), the data is described in the next section.


In [ ]:
# Import section
import time

import pandas as pd

import bfportf as bf

pd.set_option('display.precision', 3)


### Load data

`prices.parquet` (4.5 MB, 500 symbols, `float32`) is published as a GitHub release asset and is not
tracked in git: the repository stays small. The cell below uses the local copy from `assets/` when it
exists and downloads the release asset otherwise.


In [ ]:
# Load data
from pathlib import Path
import urllib.request

import pandas as pd

# prices.parquet (4.5 MB, 500 symbols) is published as a GitHub release asset and is not tracked in git.
# A local copy is preferred: the repository layout (assets/prices.parquet), a file next to the notebook,
# or ../assets when the kernel runs inside the examples directory.
PRICES_CANDIDATES = (
    Path('assets/prices.parquet'),
    Path('prices.parquet'),
    Path('../assets/prices.parquet'),
)
PRICES_URL = 'https://github.com/answering007/bfportf/releases/latest/download/prices.parquet'

local_path = next((path for path in PRICES_CANDIDATES if path.exists()), None)

if local_path is None:
    # No local copy: take the release asset and cache it next to the notebook
    local_path = Path('assets/prices.parquet') if Path('assets').is_dir() else Path('prices.parquet')
    try:
        with urllib.request.urlopen(PRICES_URL, timeout=120) as response:
            local_path.write_bytes(response.read())
    except Exception as error:
        raise RuntimeError(
            f'cannot download {PRICES_URL}, get the file manually: curl -L -o {local_path} {PRICES_URL}'
        ) from error
    source = f'{PRICES_URL} -> cached to {local_path}'
else:
    source = local_path

prices = pd.read_parquet(local_path)
if 'Date' in prices.columns:  # the file was written without an index
    prices = prices.set_index(pd.to_datetime(prices['Date'])).drop(columns=['Date'])

returns = bf.load_returns(prices)
bf.validate_returns(returns)
print(f'{returns.shape[0]} periods x {returns.shape[1]} symbols <- {source}')
returns.head()


### Sequential split

`split_sequential` walks through the history with a fixed step: a train window is followed by a test window, then
the pair moves forward by `test_days`. The test windows tile the series without overlapping, so every period is
tested at most once and the test window is always in the future of its own train window. The tail that does not
fill a complete pair stays out of the loop.


In [ ]:
def split_sequential(data: pd.DataFrame, train_days: int, test_days: int) -> list:
    """Split a time series into consecutive (train, test) windows.

    The windows move forward by ``test_days``, so the test windows do not overlap and no period is used
    for testing twice.

    Args:
        data (pd.DataFrame): Returns dataframe ordered by time.
        train_days (int): Number of periods used to find a portfolio.
        test_days (int): Number of periods used to test it.

    Returns:
        list: ``list[tuple[pd.DataFrame, pd.DataFrame]]`` of (train, test) pairs.

    Raises:
        bf.DataValidationError: If a window size is invalid or the pair does not fit into the data.
    """
    if train_days < 2:
        raise bf.DataValidationError(f'train_days must be >= 2, got {train_days}')
    if test_days < 1:
        raise bf.DataValidationError(f'test_days must be >= 1, got {test_days}')
    if train_days + test_days > len(data):
        raise bf.DataValidationError(
            f'train_days + test_days must fit into {len(data)} periods, got {train_days} + {test_days}'
        )

    segments = []
    start = 0
    while start + train_days + test_days <= len(data):
        stop = start + train_days
        segments.append((data.iloc[start:stop], data.iloc[stop:stop + test_days]))
        start += test_days
    return segments


In [ ]:
# One trading year to find a portfolio, one trading quarter to test it
TRAIN_DAYS = 30*3
TEST_DAYS = 30

segments = split_sequential(returns, train_days=TRAIN_DAYS, test_days=TEST_DAYS)
unused = len(returns) - TRAIN_DAYS - TEST_DAYS * len(segments)

segments_frame = pd.DataFrame(
    {
        'segment': range(1, len(segments) + 1),
        'train_start': [train.index[0].date() for train, _ in segments],
        'train_end': [train.index[-1].date() for train, _ in segments],
        'test_start': [test.index[0].date() for _, test in segments],
        'test_end': [test.index[-1].date() for _, test in segments],
    }
)

print(f'{len(segments)} segments x ({TRAIN_DAYS} train + {TEST_DAYS} test) periods, {unused} periods left out')
segments_frame.head()


In [ ]:
# The window sizes are validated before the loop starts
for train_days, test_days in ((1, TEST_DAYS), (TRAIN_DAYS, 0), (len(returns), TEST_DAYS)):
    try:
        split_sequential(returns, train_days=train_days, test_days=test_days)
    except bf.DataValidationError as error:
        print(f'train_days={train_days}, test_days={test_days} -> {error}')


### Walk-forward loop

Every segment repeats the same four steps, nothing is shared between the segments:

1. the inverse volatility shares are estimated on the train window (`scale_return_data`),
2. the test window is scaled with the same shares, so the test never uses information from its own future,
3. the search picks the portfolio with the best Sharpe ratio **in-sample**,
4. the portfolio and the SPY benchmark are evaluated on the untouched test window.

The out-of-sample returns are accumulated: every segment appends its `test_days` periods to the final series.


In [ ]:
started = time.perf_counter()

segment_rows = []
portfolio_returns = []
benchmark_returns = []

for number, (train, test) in enumerate(segments, start=1):
    train_scaled, weights = bf.scale_return_data(train)
    test_scaled = test * weights

    best = bf.find_portfolios(
        returns=train_scaled,
        metric='sharpe',
        custom_weights=weights,
        max_number=3,
        verbose=False,
    )[0]

    # Out-of-sample returns of the found portfolio and of the SPY benchmark
    portfolio = bf.get_portfolio_return(test_scaled, best.symbols, weights)
    benchmark = bf.get_portfolio_return(test_scaled, ['SPY'], weights)

    portfolio_returns.append(portfolio)
    benchmark_returns.append(benchmark)
    segment_rows.append(
        {
            'segment': number,
            'test_end': test.index[-1].date(),
            'symbols': best.portfolio,
            'in_sample_sharpe': best.value,
            'test_pnl': bf.pnl_target(portfolio.to_numpy()),
            'spy_pnl': bf.pnl_target(benchmark.to_numpy()),
        }
    )

print(f'{len(segments)} segments processed in {time.perf_counter() - started:.1f} s')

segments_result = pd.DataFrame(segment_rows)
segments_result['edge'] = segments_result['test_pnl'] - segments_result['spy_pnl']
segments_result.head(10)


In [ ]:
edges = segments_result['edge']
beats = int((edges > 0).sum())

print(f'portfolio beat SPY in {beats} of {len(edges)} segments ({beats / len(edges):.1%})')
print(f'median edge: {edges.median():.2%}, mean edge: {edges.mean():.2%}')
segments_result.head(20)


### The stitched out-of-sample series

The test windows do not overlap, so the segment series can be concatenated directly: the result is one continuous
out-of-sample track record of the search. `create_report` compares it with the benchmark and adds the `Diff` row
wise difference.


In [ ]:
oos = pd.concat(
    [pd.concat(portfolio_returns).rename('Portfolio'), pd.concat(benchmark_returns).rename('Benchmark')],
    axis=1,
)

print(f'{len(oos)} out-of-sample periods: {oos.index[0].date()} .. {oos.index[-1].date()}')
oos.head()


In [ ]:
formatting = {'PnL': '{:.2%}', 'CAGR': '{:.2%}', 'Max drawdown': '{:.2%}'}

bf.create_report(
    returns=oos,
    portfolio_symbols=['Portfolio'],
    benchmark_symbols=['Benchmark'],
    custom_formatting=formatting,
)


### Equity curve

Compounding the stitched returns gives the growth of one unit of capital. The benchmark is compounded on exactly
the same periods, so the two curves are directly comparable.


In [ ]:
equity = (1 + oos).cumprod()

axes = equity.plot(
    figsize=(10, 5),
    grid=True,
    title='Out-of-sample growth of 1, stitched segments',
    xlabel='date',
    ylabel='growth of 1',
)


In [ ]:
# Total out-of-sample result: one unit of capital compounded over all segments
equity.iloc[-1].sub(1).to_frame('total PnL')


### The horizon changes the result

Both window sizes are parameters, and the out-of-sample result depends on them: a longer train window gives the
search more data but reacts slower to the market, a longer test window means fewer rebalances. A small universe
keeps this cell fast, the function and the loop are exactly the same, only `train_days` and `test_days` change.


In [ ]:
# A small universe keeps this cell fast: the same loop with three different horizons
universe = returns[['SPY', 'AAPL', 'MSFT', 'NVDA', 'GLD', 'TLT', 'XOM', 'JPM', 'KO', 'AMD', 'BA', 'QQQ']]

rows = []
for train_days, test_days in ((126, 21), (252, 63), (504, 63)):
    segment_portfolio = []
    segment_benchmark = []
    for train, test in split_sequential(universe, train_days=train_days, test_days=test_days):
        train_scaled, weights = bf.scale_return_data(train)
        test_scaled = test * weights
        best = bf.find_portfolios(
            returns=train_scaled,
            metric='sharpe',
            custom_weights=weights,
            max_number=3,
            verbose=False,
        )[0]
        segment_portfolio.append(bf.get_portfolio_return(test_scaled, best.symbols, weights))
        segment_benchmark.append(bf.get_portfolio_return(test_scaled, ['SPY'], weights))

    portfolio_pnl = float(bf.pnl_target(pd.concat(segment_portfolio).to_numpy()))
    spy_pnl = float(bf.pnl_target(pd.concat(segment_benchmark).to_numpy()))
    rows.append(
        {
            'train_days': train_days,
            'test_days': test_days,
            'segments': len(segment_portfolio),
            'portfolio_pnl': portfolio_pnl,
            'spy_pnl': spy_pnl,
            'edge': portfolio_pnl - spy_pnl,
        }
    )

pd.DataFrame(rows).round(4)


### What the example shows

- The in-sample Sharpe ratio is the target of the search, not a promise: the portfolios look convincing on their
train window and the out-of-sample segments are much noisier, especially on a 500 symbol universe.
- A walk-forward loop is the cheapest honest check of a search based strategy: the test window is always in the
future, and the weights and the scaling factors come from the train window only.
- The result depends on the horizon, on the universe and on the metric, so a single train/test split of example
`01` is a point estimate, not the whole picture.

Limitations of the example: no transaction costs and no slippage, the portfolio is rebalanced at the start of
every test window, the universe is fixed (only the symbols that exist today, i.e. a survivorship bias), and the
tail of the history that does not fill a complete train + test pair is dropped.

The next steps are up to the reader: example `02` shows a custom target function, example `03` the metric
registry, the `tolerance` and the alternative search strategies.
